In [1]:
import numpy as np
import pandas as pd
import re
import ftfy
import html
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import f1_score, classification_report

In [2]:
df=pd.read_csv("development.csv",delimiter=",", index_col="Id")

### *Source* feature inspection

In [ ]:
def clean_text_light_title(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = ftfy.fix_text(text)
    text = html.unescape(text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\.\,\-\%\$\€\£]", " ", text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_text_light_article(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = ftfy.fix_text(text)
    text = html.unescape(text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\b(a\s+href|href|img\s+src|nbsp|read\s+more|click\s+here)\b',
                  ' ', text, flags=re.IGNORECASE)
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\.\,\-\%\$\€\£]", " ", text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def build_features(df_raw: pd.DataFrame) -> pd.DataFrame:
    df = df_raw.copy()

    # --- source cleaning (versione minima; OHE gestisce rarità/unknown) ---
    if "source" not in df.columns:
        df["source"] = "Other"
    df["source"] = df["source"].replace("\\N", np.nan)
    df["source"] = df["source"].replace("", np.nan)
    df["source"] = df["source"].fillna("Other").astype(str)

    # --- page_rank numeric ---
    if "page_rank" not in df.columns:
        df["page_rank"] = 0.0
    df["page_rank"] = pd.to_numeric(df["page_rank"], errors="coerce").fillna(0.0)

    # --- timestamp -> features ---
    if "timestamp" in df.columns:
        dt = pd.to_datetime(df["timestamp"], errors="coerce")
    else:
        dt = pd.to_datetime(pd.Series([pd.NaT] * len(df), index=df.index), errors="coerce")

    df["has_date"] = dt.notna().astype(int)
    df["quarter"] = dt.dt.quarter.fillna(-1).astype(int)
    df["is_weekend"] = dt.dt.dayofweek.isin([5, 6]).fillna(False).astype(int)

    # --- title/article cleaning ---
    if "title" not in df.columns:
        df["title"] = ""
    if "article" not in df.columns:
        df["article"] = ""

    title_clean = df["title"].apply(clean_text_light_title)
    article_raw = df["article"].replace("\\N", "").fillna("").astype(str)
    article_clean = article_raw.apply(clean_text_light_article)

    # --- build final text (mantengo ESATTAMENTE la tua formula, anche se ripete title 2 volte) ---
    df["text"] = (title_clean + " " + title_clean + " [TITLE] " + article_clean).str.strip()

    # Ritorna solo le feature usate dal modello
    return df[["text", "source", "page_rank", "has_date", "is_weekend", "quarter"]]

In [4]:
TEXT_COL = "text"
CAT_COLS = ["source"]
NUM_COLS = ["page_rank", "has_date", "is_weekend", "quarter"]
TARGET_COL = "label"

# df_raw: leggi il CSV GREZZO (non preprocessare a mano)
df_raw = pd.read_csv("development.csv", delimiter=",", index_col="Id")

y = df_raw[TARGET_COL].copy()

X_train, X_val, y_train, y_val = train_test_split(
    df_raw, y, test_size=0.2, random_state=42, stratify=y
)

feature_step = FunctionTransformer(build_features, validate=False)

preprocess = ColumnTransformer(
    transformers=[
        ("tfidf", TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=3,
            max_df=0.95,
            max_features=150_000,
            sublinear_tf=True
        ), TEXT_COL),
        ("source_ohe", OneHotEncoder(
            handle_unknown="infrequent_if_exist",
            min_frequency=5,
            sparse_output=True
        ), CAT_COLS),
        ("num", "passthrough", NUM_COLS),
    ],
    remainder="drop"
)

pipe = Pipeline([
    ("features", feature_step),
    ("prep", preprocess),
    ("clf", LinearSVC(class_weight="balanced", random_state=42))
])

param_dist = {
    "clf__C": np.logspace(-3, 1, 10),
    "clf__loss": ["hinge", "squared_hinge"]
}

search = RandomizedSearchCV(
    pipe,
    param_distributions=param_dist,
    n_iter=10,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1,
    random_state=42
)

search.fit(X_train, y_train)

best_model = search.best_estimator_
print("Best params:", search.best_params_)
print("Best CV Macro F1:", search.best_score_)

y_val_pred = best_model.predict(X_val)
print("Validation Macro F1:", f1_score(y_val, y_val_pred, average="macro"))
print(classification_report(y_val, y_val_pred, digits=4))

/var/folders/_r/4kq58czx6cl9n68zbg2nc56r0000gn/T/ipykernel_92061/860952313.py:48: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
/var/folders/_r/4kq58czx6cl9n68zbg2nc56r0000gn/T/ipykernel_92061/860952313.py:48: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
/var/folders/_r/4kq58czx6cl9n68zbg2nc56r0000gn/T/ipykernel_92061/860952313.py:48: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
/var/folders/_r/4kq58czx6cl9n68zbg2nc56r0000gn/T/ipykernel_92061/860952313.py:48: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure 

Best params: {'clf__loss': 'squared_hinge', 'clf__C': 0.1668100537200059}
Best CV Macro F1: 0.6936986420017556
Validation Macro F1: 0.7011096890287439
              precision    recall  f1-score   support

           0     0.7387    0.7481    0.7434      4709
           1     0.7282    0.7993    0.7621      2118
           2     0.8194    0.8109    0.8151      2232
           3     0.6073    0.5023    0.5498      1995
           4     0.7731    0.9359    0.8467      1715
           5     0.5724    0.4600    0.5101      2611
           6     0.5819    0.8194    0.6805       620

    accuracy                         0.7089     16000
   macro avg     0.6887    0.7251    0.7011     16000
weighted avg     0.7027    0.7089    0.7023     16000



In [ ]:
# Fit finale su TUTTO development
best_model.fit(df_raw, y)

# Leggi evaluation GREZZO (index_col="Id" così l'Id è l'indice)
df_eval = pd.read_csv("evaluation.csv", delimiter=",", index_col="Id")

pred_eval = best_model.predict(df_eval)

# Output richiesto: colonna 'Id' e colonna 'Predicted'
out = pd.DataFrame({
    "Id": df_eval.index,
    "Predicted": pred_eval
})

out.to_csv("submission.csv", index=False)
print(out.head())

   Id  Predicted
0   0          5
1   1          2
2   2          5
3   3          0
4   4          0
